# Analytics Village — CH01: Analytics Workbook
### Build Your Own Analytics Tables from Raw Transactions

---

This notebook teaches you to **derive analytics insights** from raw transaction data. You'll build:

1. **Customer purchase history** — visit frequency, spend patterns, recency
2. **Cohort retention analysis** — track groups of customers over time
3. **Churn detection** — identify customers who stopped visiting
4. **Basket analysis** — what products are bought together
5. **Demand patterns** — seasonality, day-of-week, payday effects
6. **Customer segmentation** — group customers by behavior

Works with either `village_normalized.db` or `village_star.db`.

## Setup

In [ ]:
!pip install pandas tabulate matplotlib -q
import os
if not os.path.exists('analytics-village'):
    !git clone https://github.com/thanachart/analytics-village.git
else:
    !cd analytics-village && git pull -q

import sys
sys.path.insert(0, 'analytics-village')

import pandas as pd
import sqlite3

# Connect directly to the database
conn = sqlite3.connect('analytics-village/challenges/ch01/data/village_normalized.db')

# Quick check
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print('Tables:', tables['name'].tolist())

---
## 1. Customer Purchase History

Build a customer-level summary from raw transactions. This is the foundation for all customer analytics.

In [ ]:
# Build customer purchase history from transactions
customer_history = pd.read_sql('''
    SELECT 
        t.customer_id,
        c.income_bracket,
        c.household_size,
        COUNT(DISTINCT t.transaction_id) AS total_visits,
        SUM(t.total_amount_thb) AS total_spend,
        AVG(t.total_amount_thb) AS avg_basket,
        MIN(t.transaction_date) AS first_visit,
        MAX(t.transaction_date) AS last_visit,
        AVG(t.satisfaction_score) AS avg_satisfaction
    FROM transactions t
    JOIN customers c ON t.customer_id = c.customer_id
    GROUP BY t.customer_id
    ORDER BY total_spend DESC
''', conn)

# Calculate days since last visit (recency)
max_date = pd.read_sql("SELECT MAX(transaction_date) AS d FROM transactions", conn).iloc[0]['d']
customer_history['last_visit'] = pd.to_datetime(customer_history['last_visit'])
customer_history['recency_days'] = (pd.to_datetime(max_date) - customer_history['last_visit']).dt.days

print(f'Customers: {len(customer_history)}')
print(f'Avg visits: {customer_history["total_visits"].mean():.1f}')
print(f'Avg spend: {customer_history["total_spend"].mean():,.0f} THB')
customer_history.head(10)

---
## 2. Cohort Retention Analysis

Group customers by their first purchase week and track how many return in subsequent weeks.

In [ ]:
# Get each customer's first visit week and all visit weeks
visits = pd.read_sql('''
    SELECT customer_id, transaction_date,
           CAST((julianday(transaction_date) - julianday((SELECT MIN(transaction_date) FROM transactions))) / 7 AS INT) AS week_number
    FROM transactions
''', conn)

# First visit week per customer
first_week = visits.groupby('customer_id')['week_number'].min().reset_index()
first_week.columns = ['customer_id', 'cohort_week']

# Merge and calculate weeks since cohort
cohort_data = visits.merge(first_week, on='customer_id')
cohort_data['weeks_since_cohort'] = cohort_data['week_number'] - cohort_data['cohort_week']

# Build retention matrix
cohort_counts = cohort_data.groupby(['cohort_week', 'weeks_since_cohort'])['customer_id'].nunique().reset_index()
cohort_sizes = cohort_data.groupby('cohort_week')['customer_id'].nunique().reset_index()
cohort_sizes.columns = ['cohort_week', 'cohort_size']

retention = cohort_counts.merge(cohort_sizes, on='cohort_week')
retention['retention_pct'] = (retention['customer_id'] / retention['cohort_size'] * 100).round(1)

# Pivot to matrix
retention_matrix = retention.pivot(index='cohort_week', columns='weeks_since_cohort', values='retention_pct')
print('Cohort Retention Matrix (% returning each week):')
retention_matrix

---
## 3. Churn Detection

A customer is "at risk" if they haven't visited in 14+ days. "Churned" if absent 30+ days.

Build this from transaction recency — no pre-built lifecycle table needed.

In [ ]:
# Churn detection from recency
churn_analysis = customer_history[['customer_id', 'income_bracket', 'total_visits', 
                                    'total_spend', 'avg_basket', 'recency_days', 'avg_satisfaction']].copy()

# Define states based on recency
def classify_state(recency):
    if recency <= 7:
        return 'active'
    elif recency <= 14:
        return 'cooling'
    elif recency <= 30:
        return 'at_risk'
    else:
        return 'churned'

churn_analysis['state'] = churn_analysis['recency_days'].apply(classify_state)

# Summary
state_summary = churn_analysis.groupby('state').agg(
    customers=('customer_id', 'count'),
    avg_spend=('total_spend', 'mean'),
    avg_visits=('total_visits', 'mean'),
    avg_recency=('recency_days', 'mean'),
).round(1)

print('Customer Lifecycle States (derived from recency):')
state_summary

In [ ]:
# Who are the at-risk and churned customers?
at_risk = churn_analysis[churn_analysis['state'].isin(['at_risk', 'churned'])]
print(f'{len(at_risk)} customers at risk or churned:')
at_risk.sort_values('total_spend', ascending=False).head(10)

---
## 4. Basket Analysis

Find which products are frequently bought together.

In [ ]:
# Product co-occurrence in the same transaction
basket_pairs = pd.read_sql('''
    SELECT a.product_id AS product_a, b.product_id AS product_b,
           COUNT(DISTINCT a.transaction_id) AS co_occurrences
    FROM transaction_items a
    JOIN transaction_items b ON a.transaction_id = b.transaction_id
        AND a.product_id < b.product_id
    WHERE a.quantity_sold > 0 AND b.quantity_sold > 0
    GROUP BY a.product_id, b.product_id
    HAVING co_occurrences >= 20
    ORDER BY co_occurrences DESC
    LIMIT 15
''', conn)

# Add product names
products = pd.read_sql('SELECT product_id, product_name FROM products', conn)
prod_map = dict(zip(products.product_id, products.product_name))
basket_pairs['product_a_name'] = basket_pairs['product_a'].map(prod_map)
basket_pairs['product_b_name'] = basket_pairs['product_b'].map(prod_map)

print('Top product pairs (bought together):')
basket_pairs[['product_a_name', 'product_b_name', 'co_occurrences']]

---
## 5. Demand Patterns

Analyze seasonality, day-of-week effects, and payday impact.

In [ ]:
# Daily revenue with calendar context
daily = pd.read_sql('''
    SELECT c.date, c.day_of_week, c.is_weekend, c.is_payday_week, c.event_name,
           COUNT(DISTINCT t.transaction_id) AS transactions,
           COALESCE(SUM(t.total_amount_thb), 0) AS revenue,
           COUNT(DISTINCT t.customer_id) AS unique_customers
    FROM calendar c
    LEFT JOIN transactions t ON c.date = t.transaction_date
    GROUP BY c.date
    ORDER BY c.date
''', conn)

# Day-of-week pattern
dow = daily.groupby('day_of_week').agg(
    avg_revenue=('revenue', 'mean'),
    avg_customers=('unique_customers', 'mean'),
    avg_transactions=('transactions', 'mean'),
).round(0)

print('Day-of-week demand pattern:')
dow

In [ ]:
# Payday effect
payday = daily.groupby('is_payday_week').agg(
    avg_revenue=('revenue', 'mean'),
    avg_transactions=('transactions', 'mean'),
    days=('date', 'count')
).round(0)
payday.index = ['Normal', 'Payday Week']

if len(payday) == 2:
    uplift = (payday.loc['Payday Week', 'avg_revenue'] / payday.loc['Normal', 'avg_revenue'] - 1) * 100
    print(f'Payday uplift: {uplift:+.1f}%')
payday

---
## 6. Customer Segmentation

Group customers by behavior using spend level and visit frequency.

In [ ]:
# Simple behavioral segmentation
seg = customer_history.copy()

# Quartile-based segmentation
seg['spend_q'] = pd.qcut(seg['total_spend'], 3, labels=['low_spend', 'mid_spend', 'high_spend'])
seg['freq_q'] = pd.qcut(seg['total_visits'], 3, labels=['low_freq', 'mid_freq', 'high_freq'])

# Cross-tabulation
segment_matrix = pd.crosstab(seg['spend_q'], seg['freq_q'], margins=True)
print('Customer Segments (spend level x visit frequency):')
segment_matrix

In [ ]:
# Segment profiles
def assign_segment(row):
    if row['spend_q'] == 'high_spend' and row['freq_q'] == 'high_freq':
        return 'Champions'
    elif row['spend_q'] == 'high_spend':
        return 'Big Spenders'
    elif row['freq_q'] == 'high_freq':
        return 'Loyal Regulars'
    elif row['state'] in ('at_risk', 'churned'):
        return 'At Risk'
    else:
        return 'Average'

seg['segment'] = seg.apply(assign_segment, axis=1)

seg_profile = seg.groupby('segment').agg(
    count=('customer_id', 'count'),
    avg_spend=('total_spend', 'mean'),
    avg_visits=('total_visits', 'mean'),
    avg_basket=('avg_basket', 'mean'),
    avg_recency=('recency_days', 'mean'),
).round(1)

print('Segment Profiles:')
seg_profile

In [ ]:
conn.close()
print('Done! Use these techniques in your challenge submission.')